In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("MySparkApp").master("local[*]").getOrCreate())

In [2]:
# Emp Data 1 & Schema

emp_data_1 = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"]
]

# Emp Data 2

emp_data_2 = [
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [3]:
emp_data_1 = spark.createDataFrame(data=emp_data_1, schema=emp_schema)
emp_data_2 = spark.createDataFrame(data=emp_data_2, schema=emp_schema)

In [4]:
emp_data_1.printSchema()
emp_data_2.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [16]:
# union and unionAll

emp = emp_data_1.union(emp_data_2)
emp.show()
"""
unionAll is an alias for union. It is used to combine two DataFrames with the same schema into a single DataFrame. 
The resulting DataFrame will contain all the rows from both DataFrames, including duplicates.
The order of the rows in the resulting DataFrame is not guaranteed to be the same as the order of the rows in the original DataFrames.
"""

# we can also union 2 Dataframes eventhough they have different schemas order using unionByName() method.
emp_by_name = emp_data_1.unionByName(emp_data_2)
emp_by_name.show()


+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [10]:
# select * from emp where salary desc
from pyspark.sql.functions import desc,asc,col
emp_sorted = emp.orderBy(col("salary").desc())
emp_sorted.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        013|          106|    Brian Kim| 45|  Male| 75000|2011-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|        015|          106|  Michael Lee| 37|  Male| 63000|2014-09-30|
|        019|          103|  Steven Chen| 36|  Male| 62000|2015-08-01|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        017|          105|  George Wang| 34|  Male| 57000|2016-03-15|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|
|        012|          105|   Susan Chen| 31|Female| 54000|2017-02-15|
|        020|          102|    Grace Kim| 32|Female| 53000|2018-11-01|
|     

In [12]:
# aggrigate functions
# select department_id, sum(salary) as total_dept_salary from emp_sorted group by department_id
from pyspark.sql.functions import sum,avg,max,min,count
emp_count = emp_sorted.groupBy("department_id").agg(sum("salary").alias("total_dept_salary"))
emp_count.show()

+-------------+-----------------+
|department_id|total_dept_salary|
+-------------+-----------------+
|          101|         165000.0|
|          107|          95000.0|
|          104|         162000.0|
|          102|         207000.0|
|          103|         232000.0|
|          106|         138000.0|
|          105|         111000.0|
+-------------+-----------------+



In [13]:
# Aggregation with having clause
# select department_id, avg(salary) as avg_dept_salary from emp_sorted  group by department_id having avg(salary) > 50000
emp_avg = emp_sorted.groupBy("department_id").agg(avg("salary").alias("avg_dept_salary")).where(col("avg_dept_salary") > 50000)
emp_avg.show()

+-------------+---------------+
|department_id|avg_dept_salary|
+-------------+---------------+
|          101|        55000.0|
|          104|        54000.0|
|          102|        51750.0|
|          103|        58000.0|
|          106|        69000.0|
|          105|        55500.0|
+-------------+---------------+



In [49]:
# how to add timezone

from pyspark.sql.functions import date_format

emp_with_timezone = emp_fixed_dates.withColumn("timezone", date_format(col("current_date"), "z"))
emp_with_timezone.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+--------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|timezone|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+--------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|     UTC|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|     UTC|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|      null|    Bob Brown|     UTC|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|     UTC|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|     UTC|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill Wong|     UTC|
|        007|          101|James Johnson| 42|  Male| 70

In [40]:
# to see all data fully
emp_with_current_dates.show(truncate=False)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |new_gender|new_name     |current_date|current_timestamp         |
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|M         |Zohn Doe     |2026-07-26  |2026-07-26 07:20:40.651971|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|F         |Zane Smith   |2026-07-26  |2026-07-26 07:20:40.651971|
|003        |102          |Bob Brown    |35 |      |55000 |2014-05-01|null      |Bob Brown    |2026-07-26  |2026-07-26 07:20:40.651971|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|F         |Alice Lee    |2026-07-26  |2026-07-26 07:20:40.651971|
|005        |103          |Jack Chan    |40 |Mal

In [43]:
# drop null values from a column
emp_dropped_nulls = emp_with_current_dates.na.drop(subset=["new_gender"])
emp_dropped_nulls.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|  2026-07-26|2026-07-26 07:21:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|  2026-07-26|2026-07-26 07:21:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-07-26|2026-07-26 07:21:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|  2026-07-26|2026-07-26 07:21:...|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill 

In [44]:
# instead of dropping null values, we can fill them with a default value using coalesce() function
from pyspark.sql.functions import coalesce, lit
emp_filled_nulls = emp_with_current_dates.withColumn("new_gender", coalesce(col("new_gender"), lit("Unknown")))
emp_filled_nulls.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|  2026-07-26|2026-07-26 07:27:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|  2026-07-26|2026-07-26 07:27:...|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|   Unknown|    Bob Brown|  2026-07-26|2026-07-26 07:27:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-07-26|2026-07-26 07:27:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack 

In [45]:
# drop old coloumns and keep only new columns
emp_final = emp_filled_nulls.drop("gender", "name").withColumnRenamed("new_gender", "gender").withColumnRenamed("new_name", "name")
emp_final.show(truncate=False)

+-----------+-------------+---+------+----------+-------+-------------+------------+--------------------------+
|employee_id|department_id|age|salary|hire_date |gender |name         |current_date|current_timestamp         |
+-----------+-------------+---+------+----------+-------+-------------+------------+--------------------------+
|001        |101          |30 |50000 |2015-01-01|M      |Zohn Doe     |2026-07-26  |2026-07-26 07:31:50.771423|
|002        |101          |25 |45000 |2016-02-15|F      |Zane Smith   |2026-07-26  |2026-07-26 07:31:50.771423|
|003        |102          |35 |55000 |2014-05-01|Unknown|Bob Brown    |2026-07-26  |2026-07-26 07:31:50.771423|
|004        |102          |28 |48000 |2017-09-30|F      |Alice Lee    |2026-07-26  |2026-07-26 07:31:50.771423|
|005        |103          |40 |60000 |2013-04-01|M      |Zack Chan    |2026-07-26  |2026-07-26 07:31:50.771423|
|006        |103          |32 |52000 |2018-07-01|F      |Zill Wong    |2026-07-26  |2026-07-26 07:31:50.

In [46]:
# saving as csv

emp_final.write.format("csv").save("data/output/4/emp_final.csv")

In [ ]:
emp_final.write.format("csv").save("data/output/2/emp.csv")